# Distributed Algorithms

This notebook demonstrates distributed computation patterns with DTL.

## Topics
- Local transformations with NumPy
- Distributed reduction patterns
- Map-reduce style computations

In [ ]:
import dtl
import numpy as np

print(f"DTL version: {dtl.__version__}")
print(f"MPI available: {dtl.has_mpi()}")

## Local Transform

Apply transformations to local data using NumPy:

In [ ]:
with dtl.Context() as ctx:
    # Create input vector
    vec = dtl.DistributedVector(ctx, size=1000, dtype=np.float64)
    local = vec.local_view()
    
    # Initialize with values 0, 1, 2, ...
    local[:] = np.arange(vec.local_offset, vec.local_offset + vec.local_size)
    
    # Apply transformation: square each element
    local[:] = local ** 2
    
    print(f"Rank {ctx.rank}: first 5 squared values = {local[:5]}")

## Distributed Sum

Compute the sum across all ranks:

In [ ]:
with dtl.Context() as ctx:
    n = 10000
    vec = dtl.DistributedVector(ctx, size=n, dtype=np.float64)
    local = vec.local_view()
    
    # All ones
    local[:] = 1.0
    
    # Local sum
    local_sum = np.sum(local)
    
    # Global sum via DTL allreduce
    global_sum = dtl.allreduce(ctx, local_sum, op=dtl.SUM)
    print(f"Global sum: {global_sum} (expected: {n})")

## Distributed Mean

Compute the mean across all ranks:

In [ ]:
with dtl.Context() as ctx:
    n = 10000
    vec = dtl.DistributedVector(ctx, size=n, dtype=np.float64)
    local = vec.local_view()
    
    # Initialize with global indices
    local[:] = np.arange(vec.local_offset, vec.local_offset + vec.local_size)
    
    # Local sum and count
    local_sum = np.sum(local)
    local_count = len(local)
    
    # Gather sums and counts using DTL allreduce
    global_sum = dtl.allreduce(ctx, local_sum, op=dtl.SUM)
    global_count = dtl.allreduce(ctx, float(local_count), op=dtl.SUM)
    
    global_mean = global_sum / global_count
    expected_mean = (n - 1) / 2  # Mean of 0, 1, ..., n-1
    
    print(f"Global mean: {global_mean}")
    print(f"Expected: {expected_mean}")

## Distributed Min/Max

In [ ]:
with dtl.Context() as ctx:
    vec = dtl.DistributedVector(ctx, size=10000, dtype=np.float64)
    local = vec.local_view()
    
    # Random data
    np.random.seed(42 + ctx.rank)
    local[:] = np.random.randn(len(local))
    
    local_min = np.min(local)
    local_max = np.max(local)
    
    # Global min/max using DTL allreduce
    global_min = dtl.allreduce(ctx, local_min, op=dtl.MIN)
    global_max = dtl.allreduce(ctx, local_max, op=dtl.MAX)
    
    print(f"Global min: {global_min:.4f}")
    print(f"Global max: {global_max:.4f}")

## Map-Reduce Pattern

A common pattern: map a function over data, then reduce:

In [ ]:
def distributed_map_reduce(ctx, vec, map_fn, reduce_op):
    """Apply map function locally, then reduce globally using DTL."""
    local = vec.local_view()
    
    # Map: apply function to local data
    local_result = map_fn(local)
    
    # Reduce: combine results across ranks using DTL
    return dtl.allreduce(ctx, local_result, op=reduce_op)

with dtl.Context() as ctx:
    n = 100000
    vec = dtl.DistributedVector(ctx, size=n, dtype=np.float64)
    local = vec.local_view()
    
    # Initialize
    local[:] = np.arange(vec.local_offset, vec.local_offset + vec.local_size)
    
    # Example 1: Sum of squares
    sum_of_squares = distributed_map_reduce(
        ctx, vec,
        map_fn=lambda x: np.sum(x ** 2),
        reduce_op=dtl.SUM
    )
    print(f"Sum of squares: {sum_of_squares:.2e}")
    
    # Example 2: Count positive values
    local[:] = np.random.randn(len(local))  # New random data
    positive_count = distributed_map_reduce(
        ctx, vec,
        map_fn=lambda x: float(np.sum(x > 0)),
        reduce_op=dtl.SUM
    )
    print(f"Positive values: {int(positive_count)} / {n} ({100*positive_count/n:.1f}%)")

## Distributed Dot Product

In [ ]:
with dtl.Context() as ctx:
    n = 10000
    
    # Create two vectors
    a = dtl.DistributedVector(ctx, size=n, dtype=np.float64)
    b = dtl.DistributedVector(ctx, size=n, dtype=np.float64)
    
    local_a = a.local_view()
    local_b = b.local_view()
    
    # Initialize: a = [1, 1, 1, ...], b = [1, 2, 3, ...]
    local_a[:] = 1.0
    local_b[:] = np.arange(a.local_offset, a.local_offset + a.local_size) + 1
    
    # Local dot product
    local_dot = np.dot(local_a, local_b)
    
    # Global dot product using DTL allreduce
    global_dot = dtl.allreduce(ctx, local_dot, op=dtl.SUM)
    expected = n * (n + 1) / 2  # Sum of 1 to n
    
    print(f"Dot product: {global_dot}")
    print(f"Expected: {expected}")

## Matrix-Vector Multiply (Distributed)

Row-partitioned matrix times replicated vector:

In [ ]:
with dtl.Context() as ctx:
    rows, cols = 1000, 100
    
    # Distributed matrix (rows partitioned)
    matrix = dtl.DistributedTensor(ctx, shape=(rows, cols), dtype=np.float64)
    local_mat = matrix.local_view()
    
    # Initialize matrix with row index
    local_mat[:, :] = np.arange(local_mat.shape[0]).reshape(-1, 1)
    
    # Replicated vector (same on all ranks)
    x = np.ones(cols)
    
    # Local matrix-vector product
    local_result = local_mat @ x
    
    print(f"Rank {ctx.rank}: computed {len(local_result)} rows")
    print(f"First 3 results: {local_result[:3]}")
    
    # Each row result equals row_index * cols
    # (since each row is filled with row_index and x is all 1s)